In [ ]:
class TemporalAnalyzer:
    def __init__(self, window_duration_sec=30, fps=30):
        """
        SG-4 Temporal Analysis Module
        window_duration_sec: Length of sliding window in seconds (e.g., 10s vs 30s for W5 experiments)
        fps: Frames per second of incoming camera stream
        """
        self.fps = fps
        self.window_size = int(window_duration_sec * fps)
        self.buffer = deque(maxlen=self.window_size)

    def process_frame(self, frame_cue):
        """
        Receives 1 per-frame cue dictionary from upstream (SG-1/2/3).
        V1 Input Contract: timestamp, face_confidence, eye_closure_prob, is_blink, yawn_prob, head_nod_event
        """
        # Validate face detection (SG-1 confidence >= 0.5)
        # Low confidence or lost face is marked invalid so looking away isn't miscounted as sleep
        is_valid = frame_cue.get("face_confidence", 0.0) >= 0.5

        processed_cue = {
            "timestamp": frame_cue.get("timestamp", time.time()),
            "is_valid": is_valid,
            "closed": is_valid and (frame_cue.get("eye_closure_prob", 0.0) >= 0.8),
            "is_blink": is_valid and frame_cue.get("is_blink", False),
            "yawn": is_valid and (frame_cue.get("yawn_prob", 0.0) >= 0.7),
            "head_nod": is_valid and frame_cue.get("head_nod_event", False)
        }
        self.buffer.append(processed_cue)

    def get_temporal_vector(self):
        """
        Computes summary fatigue indicators for SG-5 decision logic.
        V1 Output Contract: validity_score, perclos, blink_rate, microsleep_count, yawn_rate, nod_count
        """
        if not self.buffer:
            return None

        total_frames = len(self.buffer)
        valid_frames = [f for f in self.buffer if f["is_valid"]]
        num_valid = len(valid_frames)

        if num_valid == 0:
            return {
                "validity_score": 0.0,
                "perclos": 0.0,
                "blink_rate": 0.0,
                "microsleep_count": 0,
                "yawn_rate": 0.0,
                "nod_count": 0
            }

        # 1. PERCLOS (% of valid frames eyes >= 80% closed)
        closed_count = sum(1 for f in valid_frames if f["closed"])
        perclos = (closed_count / num_valid) * 100.0

        # 2. Blink Rate (Blinks per minute extrapolated over window duration)
        blink_count = sum(1 for f in valid_frames if f["is_blink"])
        window_sec = total_frames / self.fps
        blink_rate = (blink_count / window_sec) * 60.0

        # 3. Microsleep Detection (Continuous closures >= 0.5s / 15 frames)
        microsleep_count = 0
        consecutive_closed = 0
        min_microsleep_frames = int(0.5 * self.fps)

        for f in self.buffer:
            if f["is_valid"] and f["closed"]:
                consecutive_closed += 1
            else:
                if consecutive_closed >= min_microsleep_frames:
                    microsleep_count += 1
                consecutive_closed = 0
        if consecutive_closed >= min_microsleep_frames:
            microsleep_count += 1

        # 4. Facial & Head Event Counters
        nod_count = sum(1 for f in valid_frames if f["head_nod"])
        yawn_count = sum(1 for f in valid_frames if f["yawn"])
        yawn_rate = (yawn_count / window_sec) * 60.0

        return {
            "validity_score": round(num_valid / total_frames, 2),
            "perclos": round(perclos, 2),
            "blink_rate": round(blink_rate, 1),
            "microsleep_count": microsleep_count,
            "yawn_rate": round(yawn_rate, 1),
            "nod_count": nod_count
        }